In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
import pandas as pd
import numpy as np
from keras import layers
from pathlib import Path
keras.utils.set_random_seed(58008)
print("CUDA available:", torch.cuda.is_available())

I0000 00:00:1775771786.074800    7607 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775771786.098575    7607 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CUDA available: True


I0000 00:00:1775771786.567130    7607 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
torch.cuda.empty_cache()

In [3]:
train_paths = list(Path('./data/train').glob('*.jpg'))
train_df  = pd.DataFrame({
    'filepath': [str(p) for p in train_paths],
    'label': [1 if p.stem.startswith('dog') else 0 for p in train_paths]
})
train_df = train_df.sample(frac=1, random_state = 58008).reset_index(drop=True)

In [4]:
from PIL import Image
sizes = [Image.open(p).size for p in train_paths[:500]]
widths, heights = zip(*sizes)
print(f"width:  min={min(widths)} max={max(widths)} avg={sum(widths)//len(widths)}")
print(f"height: min={min(heights)} max={max(heights)} avg={sum(heights)//len(heights)}")

width:  min=66 max=500 avg=405
height: min=50 max=500 avg=362


In [5]:
IMG_SIZE = 224
BATCH_SIZE = 32 # adjust based on vram usage

In [6]:
from keras.utils import image_dataset_from_directory

from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CatDogDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        label = self.df.iloc[idx]['label']
        if self.transform:
            img = self.transform(img)
        return img, label

In [7]:
import torchvision.transforms as T
train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),  
    T.RandomGrayscale(p=0.05),                                              
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225]),
    T.RandomErasing(p=0.2),                                                 
])

split = int(.8 * len(train_df))
train_ds = CatDogDataset(train_df[:split], transform=train_transform)
val_ds = CatDogDataset(train_df[split:], transform=T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225])
]))

train_loader = DataLoader(train_ds, batch_size= BATCH_SIZE, shuffle = True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=8, pin_memory=True)

In [8]:
""" import torch.nn as nn
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
        self.silu = nn.SiLU()
    def forward(self, x):
        return self.silu(x+self.block(x)) """

' import torch.nn as nn\nclass ResBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(channels, channels, 3, padding=1),\n            nn.BatchNorm2d(channels),\n            nn.SiLU(),\n            nn.Conv2d(channels, channels, 3, padding=1),\n            nn.BatchNorm2d(channels)\n        )\n        self.silu = nn.SiLU()\n    def forward(self, x):\n        return self.silu(x+self.block(x)) '

In [9]:
import torch.nn as nn
class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // r),
            nn.SiLU(),
            nn.Linear(channels // r, channels),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        return x * self.se(x).view(-1, x.shape[1], 1, 1)

In [10]:
class MBConv(nn.Module):
    def __init__(self, channels, expand=4):
        super().__init__()
        mid = channels * expand
        self.block = nn.Sequential(
            #expand
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(),

            nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(),

            SEBlock(mid),

            nn.Conv2d(mid, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

In [11]:
class HaydenNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # 3 channels, 32 filters,
            nn.BatchNorm2d(32),
            nn.SiLU(),
            MBConv(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 inputs to match previous layer output of 32
            nn.BatchNorm2d(64),
            nn.SiLU(),
            MBConv(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            MBConv(128),
            SEBlock(128),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(.3),
            nn.Linear(64,1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [12]:
import torchvision.models as models
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device', device)

""" model = models.efficientnet_b0(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(.3),
    nn.Linear(model.classifier[1].in_features, 1),
    nn.Sigmoid()
)
model = model.to(device)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=.001)
 """

model = HaydenNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=.001)




Device cuda


In [13]:
#from torchinfo import summary
#summary(model, input_size=(1, 3, 224, 224))

In [14]:
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
scaler = GradScaler()

best_acc = 0
patience = 10
epochs_no_improve = 0

for epoch in range(100):
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/100")
    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.float().to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            preds = model(imgs).squeeze()
            loss = criterion(preds, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        loop.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    correct, total = 0, 0
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="  Validating", leave=False):
            imgs, labels = imgs.to(device), labels.float().to(device)
            with autocast():
                preds = model(imgs).squeeze()
                val_loss += criterion(preds, labels).item()
            correct += ((preds > 0) == labels).sum().item()
            total += len(labels)

    acc = correct / total
    scheduler.step(val_loss)
    print(f'Epoch {epoch+1} | Val Acc: {acc:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

    if acc > best_acc:
        best_acc = acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> saved new best ({best_acc:.4f})')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

<positron-console-cell-14>:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  Validating:   0%|          | 0/157 [00:00<?, ?it/s]<positron-console-cell-14>:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


Epoch 1 | Val Acc: 0.6798 | LR: 1.00e-03
  -> saved new best (0.6798)


Epoch 2/100: 100%|██████████| 625/625 [01:17<00:00,  8.03it/s, loss=0.3567]


Epoch 2 | Val Acc: 0.7540 | LR: 1.00e-03
  -> saved new best (0.7540)


Epoch 3/100: 100%|██████████| 625/625 [01:17<00:00,  8.03it/s, loss=0.4071]


...

Epoch 45 | Val Acc: 0.9648 | LR: 1.00e-05


Epoch 46/100: 100%|██████████| 625/625 [02:00<00:00,  5.19it/s, loss=0.0715]


Epoch 46 | Val Acc: 0.9664 | LR: 1.00e-05


Epoch 47/100: 100%|██████████| 625/625 [01:14<00:00,  8.36it/s, loss=0.0090]


Epoch 47 | Val Acc: 0.9666 | LR: 1.00e-05
Early stopping at epoch 47


In [15]:
model = HaydenNet()  # same architecture used during training
model.load_state_dict(torch.load('/home/richard/code/predictive_hw2/best_model.pth', map_location=device))
model.to(device)

HaydenNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU()
    (3): MBConv(
      (block): Sequential(
        (0): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU()
        (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=128, bias=False)
        (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): SiLU()
        (6): SEBlock(
          (se): Sequential(
            (0): AdaptiveAvgPool2d(output_size=1)
            (1): Flatten(start_dim=1, end_dim=-1)
            (2): Linear(in_features=128, out_features=8, bias=True)
            (3): SiLU()
            (4): Linear(in_features=8, out_features=128, bias=True)
            (5): Sigm

In [16]:
test_paths = sorted(Path('./data/test').glob('*jpg'), key=lambda p: int(p.stem))

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225])
])

ids, preds = [], []
model.eval()
with torch.no_grad():
    for path in test_paths:
        img = Image.open(path).convert("RGB")
        img = test_transform(img).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(img)).item()
        ids.append(int(path.stem))
        preds.append(prob)

submission = pd.DataFrame({'id': ids, 'label':preds})
submission.to_csv('submission.csv', index=False)

In [17]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  34803 KiB |   6068 MiB | 522272 GiB | 522272 GiB |
|       from large pool |  26048 KiB |   6063 MiB | 522083 GiB | 522083 GiB |
|       from small pool |   8755 KiB |      9 MiB |    188 GiB |    188 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  34803 KiB |   6068 MiB | 522272 GiB | 522272 GiB |
|       from large pool |  26048 KiB |   6063 MiB | 522083 GiB |